# 2B · Configure the LLM Gateway with Simple Setup

**Purpose:** Create a gateway profile from a preset and verify it with one chat request.

**What we're doing**

Use **Simple Setup** in the Klique WebApp to create a profile and its routing configuration. Copy the profile's endpoint and model alias, call it with a gateway token, and find the request in **AI Usage**.

**Why we're doing it**

In [02a](02a_Use_the_LLM_Gateway.ipynb), you called an existing profile. Here you create a separate one without configuring the upstream model and routing objects by hand.

**What to look for**

* A new profile, routed resource, and its upstream resources
* A nonempty reply through that profile's endpoint and alias
* The corresponding call in **AI Usage**

**Output**

A working chat completion through a profile created by Simple Setup.

> **Validation:** The wizard steps are documented by Klique. The client example has been checked offline; verify the generated profile and make one live call on your deployment before recording or relying on the outputs.

## Create a profile from a preset

**Before you start:** Use a Klique workspace with the LLM Gateway and Simple Setup enabled, permission to configure it (or an administrator's help), and Python 3 with **requests** installed. Use a test workspace or permission to create a separate profile; do not change a shared production profile for this exercise.

1. In the WebApp, go to **Control → AI Resources → LLM Gateway**. Beside **Resources, routing & profiles**, click **+ Simple Setup**.
2. Choose **Quality first**. The documented preset sends every request to Claude Opus. This makes a short first-call demo; model usage may incur provider charges.
3. In **AI resource configuration**, enter the **Anthropic API key** if prompted. If all required resources already exist, the wizard says no key is needed. Keep the provider key in the UI; don't put it in the notebook, logs, or a recording.
4. Finish the wizard. Confirm the profile, routed resource, and upstream resource on the LLM Gateway page. If the name already exists or the wizard differs, inspect the existing setup or ask your administrator before proceeding.

> **Checkpoint:** The new profile is visible, and your caller can access it. In **My Setup → Profile → Your AI Endpoints**, find its endpoint and **model alias**. The reference architecture describes an alias called **auto**; copy the alias your deployment displays rather than using the profile or routed resource name as the request model.

<details>
<summary>Other Simple Setup presets</summary>

| Preset | Documented routing |
|---|---|
| Rate limit fallback | Claude Haiku first; overflow to Sonnet at the gateway rate limit |
| Cost first | Tool-using requests to Sonnet; other requests to Haiku |
| Long context | Long prompts to Sonnet; shorter prompts to Haiku |
| Vision | Image requests to Sonnet; text-only requests to Haiku |
| Coding | Recognized coding agents to Sonnet; other requests to Haiku |

These are the built-in Anthropic presets. The resulting configuration can be edited afterward. The gateway's own limits and a provider's rate-limit errors are different; see the setup docs for fallback behavior.

</details>

<!-- screenshot: images/llm_gateway_simple_setup.png (Preset, AI resource configuration with secret hidden, then resulting profile.) -->

## Enter the endpoint and alias

Copy the profile's OpenAI-compatible endpoint **from your own trusted Klique WebApp**, preserving any profile path. If it shows a base URL ending in **/v1**, append **/chat/completions**; if it already shows the chat completions URL, use it as-is. For the shared gateway URL used in 02a (**…/llm/v1/chat/completions**), set **PROFILE_HEADER** to the exact profile name too. Leave that field blank when the copied URL itself selects the profile.

The notebook uses your **AI Application Gateway token**, not the Anthropic key. Load it from the environment, reuse **KLIQUE_GATEWAY_TOKEN** in the course's **.env** file, or enter it at the masked prompt. Use a token authorized for this profile.

In [ ]:
import os
from getpass import getpass
from pathlib import Path
from urllib.parse import urlsplit

import requests

CHAT_URL = ""        # Copy the chat URL; preserve any profile-specific path.
MODEL_ALIAS = ""     # Copy the alias, not the upstream model or profile name.
PROFILE_HEADER = ""  # Set only when using the shared gateway endpoint from 02a.

CHAT_URL = CHAT_URL.strip().rstrip("/")
MODEL_ALIAS = MODEL_ALIAS.strip()
PROFILE_HEADER = PROFILE_HEADER.strip()
endpoint = urlsplit(CHAT_URL)
if (
    endpoint.scheme != "https"
    or not endpoint.hostname
    or endpoint.username
    or endpoint.password
    or endpoint.query
    or endpoint.fragment
    or not endpoint.path.endswith("/chat/completions")
    or not MODEL_ALIAS
):
    raise ValueError("Set the trusted HTTPS chat completions URL and exact model alias first.")


def load_gateway_token():
    key = "KLIQUE_GATEWAY_TOKEN"
    value = os.environ.get(key, "").strip()
    if value:
        return value
    for folder in (Path.cwd(), Path.cwd().parent):
        env_file = folder / ".env"
        if env_file.is_file():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                name, sep, raw = line.strip().partition("=")
                if sep and name.strip() == key:
                    value = raw.strip().strip("'\"")
                    if value:
                        return value
    return getpass("AI Application Gateway token: ").strip()


token = load_gateway_token()
if not token:
    raise ValueError("A gateway token is required.")
print("Gateway token loaded (not shown).")

## Send one chat request

Send a short prompt through the copied alias. For a failed request, the cell reports the HTTP status without printing headers, credentials, or raw upstream error text.

In [ ]:
from datetime import datetime, timezone

headers = {"Authorization": f"Bearer {token}"}
if PROFILE_HEADER:
    headers["X-ClearML-Profile"] = PROFILE_HEADER

sent_at = datetime.now(timezone.utc)
try:
    response = requests.post(
        CHAT_URL,
        headers=headers,
        json={
            "model": MODEL_ALIAS,
            "messages": [
                {"role": "user", "content": "Explain an LLM gateway in one sentence."}
            ],
            "max_tokens": 256,
        },
        timeout=60,
        allow_redirects=False,
    )
except requests.exceptions.RequestException as exc:
    raise RuntimeError(f"Gateway request failed: {type(exc).__name__}.") from None

if not 200 <= response.status_code < 300:
    raise RuntimeError(
        f"Gateway returned HTTP {response.status_code}. See Troubleshooting below."
    )

try:
    reply = response.json()
    answer = reply["choices"][0]["message"]["content"]
except (ValueError, KeyError, TypeError, IndexError):
    raise RuntimeError("The gateway did not return a chat completion.") from None
if not isinstance(answer, str) or not answer.strip():
    raise RuntimeError("No text returned. Check the model and available output tokens.")

usage = reply.get("usage") or {}
print(f"Request time (UTC): {sent_at.isoformat(timespec='seconds')}")
print(f"Requested alias:    {MODEL_ALIAS}")
print(f"Reported model:     {reply.get('model') or 'not reported'}")
print(f"Total tokens:       {usage.get('total_tokens', 'not reported')}")
print(f"Reply: {answer}")

## Verify in AI Usage

> **Checkpoint:** A successful request returns a nonempty answer. In the WebApp, open **Overview → AI Usage** if you have access, or ask an administrator to check it. Match the request by its UTC time, user, profile or alias, and token count where available. Check the recorded upstream model or route if you need to prove which backend served it.

A single successful call does not verify how unknown aliases or unauthorized users are handled, nor does it test routing limits, fallbacks, or budgets.

<!-- screenshot: images/llm_gateway_simple_setup_ai_usage.png (Hide unrelated usage and identifiers.) -->

## Troubleshooting

| Symptom | Check |
|---|---|
| Authentication or permission error | Verify the gateway token is valid and its user can access the generated profile. The provider key belongs in the WebApp. |
| Endpoint or model error | Recopy the full chat URL and alias from **Your AI Endpoints**. If using the shared URL from 02a, check the profile header. |
| HTTP 429 | Check gateway limits, wallet policy, and provider quotas before retrying. |
| HTTP 5xx or timeout | Check the upstream resource configuration, provider credentials, and network reachability. |
| An answer but no matching usage record | Verify dashboard access, the time filter and timezone, and the profile or user shown. |

**Next:** [02c Models and Routed Resources](02c_Models_and_Routed_Resources.ipynb) compares models and shows routing based on request attributes. Follow its own setup instructions; the single-model preset does not provide its multi-model test configuration.

## Related Docs

* [Simple Setup](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_llm_gateway#simple-setup)
* [LLM Gateway and Tokenomics Reference Architecture](https://klique.ai/resources/reference-architectures/llm-gateway/)
* [AI Usage](https://klique.ai/docs/latest/docs/webapp/webapp_admin_dashboard_ai_usage)